# 1. Project Introduction: Diabetes Progression Prediction

### Goal
The objective of this project is to build a regression model to predict diabetes disease progression in patients one year after baseline, based on several numerical features such as age, BMI, and blood pressure.

### Problem Type
This is a **Regression** problem, as we are predicting a continuous quantitative measure of disease progression.

### Exploratory Starting Point
This notebook represents the initial exploratory phase of a data scientist's workflow. The focus here is on understanding the data, validating its quality, performing basic cleaning, and testing baseline models.

### Path to MLOps
Once this exploratory work is complete, the code will be refactored into a structured MLOps repository. This future repository will include:
- **Data Extraction & Validation scripts**
- **Automated Cleaning & Preprocessing**
- **Experiment Tracking with MLflow**
- **Model Serving via FastAPI**
- **Containerization with Docker**
- **CI/CD with GitHub Actions**

## 2. Environment and Imports

We will use standard libraries for data manipulation, visualization, and machine learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Set plotting style
sns.set_theme(style="whitegrid")

In [ ]:
import torch
import tensorflow as tf
print("pandas:", pd.__version__)
print("GPU PyTorch:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("GPU TensorFlow:", tf.config.list_physical_devices('GPU'))

## 3. Load the Dataset

We use the built-in Scikit-learn Diabetes dataset.

In [ ]:
diabetes = load_diabetes()

# Create a pandas DataFrame for features
df = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)

# Add the target column
df['target'] = diabetes.target

print(f"Dataset Shape: {df.shape}")
print(f"Column Names: {df.columns.tolist()}")
display(df.head())

## 4. Basic Data Exploration

Understanding the distribution and quality of our data.

In [ ]:
# Summary statistics and info
print("Data Types and Missing Values:")
display(df.info())

print("\nSummary Statistics:")
display(df.describe())

# Check for duplicates
print(f"\nDuplicate rows found: {df.duplicated().sum()}")

# Visualize Target Distribution
plt.figure(figsize=(8, 4))
sns.histplot(df['target'], kde=True)
plt.title("Distribution of Diabetes Progression (Target)")
plt.show()

# Correlation with target
correlations = df.corr()['target'].sort_values(ascending=False)
print("\nCorrelation with Target Variable:")
print(correlations)

## 5. Data Extraction Logic

In this notebook, we load the data directly from `sklearn`. However, in a production MLOps setup, this logic will be moved to a **Data Extraction Script** that might pull data from a SQL database, a cloud bucket (S3/GCS), or an external API.

## 6. Data Validation Checks

Before training, we verify the data meets our expectations. These checks will eventually form a **Validation Script**.

In [ ]:
def validate_data(dataframe):
    # 1. Check if target exists
    assert 'target' in dataframe.columns, "Target column missing!"

    # 2. Check for required features
    required_features = ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']
    for col in required_features:
        assert col in dataframe.columns, f"Feature {col} is missing!"

    # 3. Ensure all columns are numeric
    assert dataframe.select_dtypes(include=[np.number]).shape[1] == dataframe.shape[1], "Non-numeric data detected!"

    # 4. Critical Missing Values
    assert dataframe.isnull().sum().sum() == 0, "Critical missing values found!"

    # 5. Dataset size
    assert len(dataframe) > 0, "Dataset is empty!"

    print("✅ Data Validation Passed!")

validate_data(df)

## 7. Data Cleaning

Even though the Scikit-learn dataset is clean, we demonstrate the cleaning workflow that will be automated in the final project.

In [ ]:
def clean_data(input_df):
    # Create a copy to avoid side effects
    cleaned_df = input_df.copy()

    # Handle missing values (demonstration)
    cleaned_df = cleaned_df.dropna()

    # Remove duplicates
    cleaned_df = cleaned_df.drop_duplicates()

    # Ensure types are correct
    for col in cleaned_df.columns:
        cleaned_df[col] = pd.to_numeric(cleaned_df[col])

    return cleaned_df

print(f"Shape before cleaning: {df.shape}")
df_cleaned = clean_data(df)
print(f"Shape after cleaning: {df_cleaned.shape}")

## 8. Feature and Target Split

We separate the input features (X) from the target progression value (y).

In [ ]:
X = df_cleaned.drop(columns=['target'])
y = df_cleaned['target']

print("X represents our patient features.")
print("y represents the progression measure.")

## 9. Train/Test Split

We split the data to evaluate the model on unseen data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

## 10. Baseline Model Training

We compare three simple regression algorithms.

In [ ]:
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)

    results.append({"Model": name, "RMSE": rmse, "MAE": mae, "R2": r2})

results_df = pd.DataFrame(results)
display(results_df)

## 11. Model Evaluation Results

We identify the best performing model based on the Root Mean Squared Error (RMSE).

In [ ]:
best_model_row = results_df.loc[results_df['RMSE'].idxmin()]
print(f"The best baseline model is {best_model_row['Model']} with an RMSE of {best_model_row['RMSE']:.2f}")

## 12. Simple Feature Engineering Exploration

Let's see if creating interaction terms between features improves our results.

In [ ]:
df_eng = df_cleaned.copy()

# 1. Interaction between BMI and Age
df_eng['bmi_age_interaction'] = df_eng['bmi'] * df_eng['age']

# 2. Interaction between Blood Pressure and BMI
df_eng['bp_bmi_interaction'] = df_eng['bp'] * df_eng['bmi']

# 3. Ratio of serum measurements (checking for division by zero)
df_eng['s1_s2_ratio'] = df_eng['s1'] / (df_eng['s2'] + 1e-9)

# Split and train again
X_eng = df_eng.drop(columns=['target'])
y_eng = df_eng['target']
X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(X_eng, y_eng, test_size=0.2, random_state=42)

eng_results = []
for name, model in models.items():
    model.fit(X_train_e, y_train_e)
    preds = model.predict(X_test_e)
    rmse = np.sqrt(mean_squared_error(y_test_e, preds))
    eng_results.append({"Model": name + "_Eng", "RMSE": rmse})

display(pd.DataFrame(eng_results))

## 13. Final Data Science Conclusion

### Findings
- **Best Model:** Based on our exploration, the **LinearRegression** (or specified best) performed best on the base features.
- **Feature Engineering:** The interactions created (BMI*Age, BP*BMI) provided a minor/significant change in RMSE, showing that feature relationships are worth exploring.
- **Metrics:** We prioritized **RMSE** to penalize larger errors, though MAE and R2 were monitored to ensure consistency.
- **Preparation:** The data is clean, but requires a structured validation step to ensure consistency when new data arrives.

### Next Steps in MLOps
The exploratory work is complete. The next step is to convert this notebook into a production-ready **structured MLOps repository** containing:
- `requirements.txt` for dependency management.
- A local virtual environment setup.
- **data_extraction.py**: For loading raw data.
- **data_validation.py**: To ensure input quality.
- **data_cleaning.py**: To handle preprocessing steps.
- **train.py**: For model training and serializing the model artifacts.
- **MLflow tracking**: To log metrics, parameters, and models.
- **Model Registry**: To select and manage the 'Champion' model.
- **FastAPI**: To serve the model via a REST API.
- **Dockerfile**: To containerize the application.
- **GitHub Actions**: To automate the CI/CD pipeline.
- **README.md**: Documenting the entire project lifecycle.